# 23.4 Databricks:湖仓平台与 Medallion 架构 / Databricks: Lakehouse Platform & Medallion Architecture

**中文**:**Databricks** 是 Spark 的创造者(UC Berkeley 的团队)创办的公司,做的是**统一数据分析平台**:把 Part 21 学的东西——托管 Spark(21.1–21.3)、Delta Lake 湖仓(21.11)、协作式 notebook、MLflow(22.8)、作业调度——**打包成一个跑在你云上(AWS/GCP/Azure)的一体化平台**。它是"湖仓一体(Lakehouse)"概念的主要推动者,在数据工程和 ML 平台市场极其流行。但比"会点 Databricks 的按钮"更重要的,是它倡导的一套**数据组织最佳实践——Medallion(奖章)架构**:把数据管道分成 **青铜(bronze)→白银(silver)→黄金(gold)** 三层,逐层提升数据质量。本节从零实现一条 medallion 流水线,亲眼看到数据如何从"原始脏数据"被逐层精炼成"可直接用于 BI/ML 的干净聚合"。
**English**: **Databricks** is the company founded by Spark's creators (the UC Berkeley team), building a **unified data-analytics platform**: it packages what Part 21 taught — managed Spark (21.1–21.3), Delta Lake lakehouse (21.11), collaborative notebooks, MLflow (22.8), job scheduling — **into an integrated platform running on your cloud (AWS/GCP/Azure)**. It is the main promoter of the "Lakehouse" concept and extremely popular in data-engineering and ML-platform markets. But more important than "clicking Databricks buttons" is the **data-organization best practice it champions — the Medallion architecture**: split the data pipeline into **bronze → silver → gold** layers, progressively improving data quality. This section builds a medallion pipeline from scratch, watching data get refined layer by layer from "raw dirty data" into "clean aggregates ready for BI/ML."

---

**中文**:**Medallion(奖章)架构 —— 数据湖仓的组织范式**。核心思想:不要一步到位地把原始数据变成分析结果,而是**分层递进、每层职责单一、数据质量逐层提升**:
**English**: **The Medallion architecture — the organizing paradigm for data lakehouses.** Core idea: don't turn raw data into analytical results in one step; instead **layer progressively, each layer with a single responsibility, data quality improving layer by layer**:
- **中文**:**青铜层(Bronze)= 原始摄取**。把源数据**原封不动**地落进来(即使很脏),只做最小改动(加个摄取时间戳)。**追加式、不可变**——它是源数据的忠实副本,出了问题能追溯、能重放。
  **Bronze = raw ingestion**. Land source data **as-is** (even if messy), with minimal change (add an ingestion timestamp). **Append-only, immutable** — a faithful copy of the source, so problems are traceable and replayable.
- **中文**:**白银层(Silver)= 清洗与规范**。去重、去空、类型转换、校验(如金额不能为负)、标准化(大小写统一)、join 关联。得到**干净、可信、结构化**的表——是数据科学家真正开始工作的地方。
  **Silver = cleaning and conforming**. Dedupe, drop nulls, cast types, validate (e.g. no negative amounts), normalize (unify casing), join. Yields **clean, trustworthy, structured** tables — where data scientists actually start working.
- **中文**:**黄金层(Gold)= 业务聚合**。按业务需求做聚合、指标计算、宽表(如"每日各渠道的成交额")——**直接喂给 BI 报表、仪表板、ML 训练**。
  **Gold = business aggregates**. Aggregate per business needs, compute metrics, build wide tables (e.g. "daily revenue per channel") — **fed directly to BI reports, dashboards, ML training**.

**中文**:为什么分层?①**职责清晰、可维护**——清洗逻辑集中在 silver,聚合逻辑集中在 gold,改一处不影响全局;②**可追溯、可重算**——bronze 保留原始数据,silver/gold 出错能从上游重建;③**复用**——多个 gold 表可以共用同一个 silver,不用重复清洗;④**质量递进**——每层是一道质量关卡。这正是 Databricks(和整个现代数据工程)推崇的组织方式。
**English**: Why layer? ① **clear responsibility, maintainable** — cleaning logic concentrated in silver, aggregation in gold, so a change in one doesn't affect everything; ② **traceable, recomputable** — bronze keeps raw data, so silver/gold errors can be rebuilt from upstream; ③ **reuse** — multiple gold tables share one silver, avoiding repeated cleaning; ④ **progressive quality** — each layer is a quality gate. This is exactly the organization Databricks (and modern data engineering) champions.

> 💡 **面试速查 / Interview cheat-sheet（★★ 数据平台/工程必考）**
> **中文**:**Databricks**=Spark 创造者做的**统一湖仓平台**:托管 Spark + **Delta Lake**(湖仓, 21.11) + 协作 notebook + MLflow + 作业调度 + Unity Catalog(数据治理) + Photon(向量化引擎), 跑在 AWS/GCP/Azure 上。**Medallion 架构**(核心最佳实践):**Bronze**(原始摄取, 原样/追加式/不可变)→**Silver**(清洗/去重/类型/校验/join, 干净可信)→**Gold**(业务聚合宽表, 供 BI/ML)。**分层价值**:职责清晰、可追溯重算、复用、质量递进。**关键组件**:Delta Lake(ACID/时间旅行/MERGE, 接 21.11)、Unity Catalog(权限/血缘/发现)、Databricks Workflows(编排, 接 23.6)、Photon(C++ 加速)、Databricks SQL(数仓)。**vs 自建 Spark**:Databricks 是托管+一体化(省运维), 但有厂商溢价和一定锁定。**vs Snowflake**:Databricks 偏数据工程+ML+湖仓(代码/notebook), Snowflake 偏 SQL 数仓+易用; 两者在互相渗透。面试金句:*"Databricks 是 Spark 团队做的统一湖仓平台(托管 Spark+Delta Lake+notebook+MLflow+调度); 核心最佳实践是 Medallion 架构——bronze 原始摄取、silver 清洗校验、gold 业务聚合, 逐层提升数据质量, 好处是职责清晰、可追溯重算、复用; 底层 Delta Lake 提供 ACID 和时间旅行, Unity Catalog 做治理。"*
> **English**: **Databricks** = a **unified lakehouse platform** by Spark's creators: managed Spark + **Delta Lake** (lakehouse, 21.11) + collaborative notebooks + MLflow + job scheduling + Unity Catalog (data governance) + Photon (vectorized engine), running on AWS/GCP/Azure. **Medallion architecture** (core best practice): **Bronze** (raw ingest, as-is/append-only/immutable) → **Silver** (clean/dedupe/type/validate/join, clean and trustworthy) → **Gold** (business aggregate wide tables, for BI/ML). **Layering value**: clear responsibility, traceable/recomputable, reuse, progressive quality. **Key components**: Delta Lake (ACID/time travel/MERGE, per 21.11), Unity Catalog (permissions/lineage/discovery), Databricks Workflows (orchestration, per 23.6), Photon (C++ acceleration), Databricks SQL (warehouse). **vs self-hosted Spark**: Databricks is managed + integrated (less ops) but with a vendor premium and some lock-in. **vs Snowflake**: Databricks leans data-engineering + ML + lakehouse (code/notebooks), Snowflake leans SQL warehouse + ease of use; the two are converging. Interview line: *"Databricks is a unified lakehouse platform by the Spark team (managed Spark + Delta Lake + notebooks + MLflow + scheduling); its core best practice is the Medallion architecture — bronze raw ingest, silver cleaning/validation, gold business aggregates, progressively improving data quality, with benefits of clear responsibility, traceability/recomputation, reuse; underneath, Delta Lake provides ACID and time travel, Unity Catalog does governance."*


In [ ]:

# ============================================================
# 从零实现 Medallion 架构流水线:bronze → silver → gold / medallion pipeline from scratch
# 中文:模拟一条真实的数据管道:原始脏数据(bronze)→ 清洗校验(silver)→ 业务聚合(gold)。逐层提升数据质量。
# English: simulate a real pipeline: raw dirty data (bronze) → cleaned/validated (silver) → business aggregates (gold). Quality improves per layer.
# ============================================================
import pandas as pd, numpy as np
# ---- BRONZE:原始摄取, 原样保留(包含各种脏数据)/ raw ingest, kept as-is (with all the mess) ----
bronze=pd.DataFrame({
    "user_id":["u1","u2","u2","u3","u4",None,"u5"],           # 有一个 null, 一条重复(u2)/ a null, a duplicate
    "event":  ["Buy","buy","buy","VIEW","view","click","buy"], # 大小写不一致 / inconsistent casing
    "amount": ["100","200","200","0","0","5","-3"],           # 字符串, 有一个负值(脏)/ strings, a negative
    "ts":     ["2024-01-01","2024-01-01","2024-01-01","2024-01-02","2024-01-02","2024-01-02","2024-01-03"]})
print(f"🥉 BRONZE 原始摄取: {len(bronze)} 行 (原样保留, 含 null/重复/负值/字符串金额/大小写乱)")

# ---- SILVER:清洗、去重、类型转换、校验、标准化 / clean, dedupe, cast, validate, normalize ----
silver=(bronze
        .dropna(subset=["user_id"])                          # 去空 / drop nulls
        .drop_duplicates(subset=["user_id","event","ts"])    # 去重 / dedupe
        .assign(event=lambda d: d["event"].str.lower(),      # 标准化大小写 / normalize casing
                amount=lambda d: pd.to_numeric(d["amount"])) # 类型转换 str→num / cast types
       )
silver=silver[silver["amount"]>=0]                           # 校验:金额不能为负 / validate: no negative amounts
print(f"🥈 SILVER 清洗校验后: {len(silver)} 行 (删除了 null/重复/负值; 类型和大小写已规范)")

# ---- GOLD:业务聚合, 供 BI/ML 直接使用 / business aggregates, ready for BI/ML ----
gold=(silver.groupby("event")
      .agg(event_count=("event","size"), total_revenue=("amount","sum"))
      .reset_index().sort_values("total_revenue",ascending=False))
print(f"🥇 GOLD 业务聚合 (供 BI 报表/仪表板/ML 训练直接使用):")
print(gold.to_string(index=False))
print("\n每层职责单一、质量递进:bronze(忠实原始, 可追溯)→silver(干净可信, DS 起点)→gold(业务就绪)")


In [ ]:

# ============================================================
# 可视化:Medallion 分层 + 数据质量递进 / medallion layers + progressive quality
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 每层行数/质量 / rows & quality per layer
layers=["🥉 Bronze\n原始摄取","🥈 Silver\n清洗校验","🥇 Gold\n业务聚合"]
rows=[len(bronze), len(silver), len(gold)]
quality=[30, 85, 100]   # 数据质量分(示意)/ illustrative data-quality score
x=np.arange(3)
ax[0].bar(x-0.2,rows,0.4,label="行数 rows",color="#4C72B0")
ax2=ax[0].twinx(); ax2.plot(x,quality,"o-",color="#C44E52",lw=2,ms=10,label="数据质量分")
ax[0].set_xticks(x); ax[0].set_xticklabels(layers,fontsize=9); ax[0].set_ylabel("行数"); ax2.set_ylabel("数据质量分",color="#C44E52")
ax[0].set_title("Medallion:行数收敛, 数据质量逐层递进"); ax2.set_ylim(0,110)
# ② 平台组件 / platform stack
ax[1].axis("off"); ax[1].set_title("Databricks 湖仓平台组件",fontsize=12,weight="bold")
comps=[("协作 Notebook + 作业调度 Workflows","#9467BD"),("MLflow(实验/模型, 22.8) + Databricks SQL","#DD8452"),
       ("Spark 引擎 + Photon(向量化加速)","#55A868"),("Delta Lake(ACID/时间旅行/MERGE, 21.11)","#4C72B0"),
       ("Unity Catalog(权限/血缘/发现) · 你的云对象存储","#8172B3")]
for i,(c,col) in enumerate(comps):
    ax[1].add_patch(plt.Rectangle((0.05,0.78-i*0.16),0.9,0.13,fc=col,alpha=0.25,ec=col,transform=ax[1].transAxes))
    ax[1].text(0.5,0.845-i*0.16,c,ha="center",va="center",fontsize=8.5,transform=ax[1].transAxes)
ax[1].text(0.5,0.02,"一体化平台:数据(Delta)+计算(Spark)+ML(MLflow)+治理(Unity)跑在你的云上",ha="center",fontsize=8,style="italic",transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/cloud04_viz.png",dpi=80); plt.show()
print("左:数据逐层精炼(行数收敛、质量递进); 右:Databricks 把 Spark/Delta/MLflow/治理打包成一体化湖仓平台")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **Medallion 架构的价值是"把混乱的数据管道变成有纪律的工程"**:现实中的数据管道很容易变成一团意大利面——原始数据在各种脚本里被随意清洗、聚合、又清洗,逻辑散落各处,出了问题无从追溯,想复用别人的清洗又不敢信。Medallion 用三层简单纪律解决它:①**bronze 忠实保留原始**(永远能追溯、能重放)、②**silver 集中清洗校验一次**(所有下游共享同一份干净数据,不用各清各的)、③**gold 按业务聚合**(直接可用)。我们的实验清楚地展示了这个"逐层精炼"的过程:7 行含 null/重复/负值/类型错乱的脏数据,经 silver 变成 4 行干净可信的记录,再到 gold 成为业务聚合。**这套分层思想不是 Databricks 独有,而是现代数据工程的通用最佳实践**——即使你不用 Databricks,用 Spark/dbt/Airflow 也应该这么组织。
2. **Databricks 的价值是"一体化 + 免运维",代价是"厂商溢价 + 一定锁定"**:它把你在 Part 21/22 学的一堆开源工具(Spark、Delta Lake、MLflow)打包成一个跑在你云上的托管平台,好处实在:①**不用自己搭 Spark 集群、调优、运维**(省一整个平台团队的活);②**数据(Delta)+ 计算(Spark)+ ML(MLflow)+ 治理(Unity Catalog)+ 协作(notebook)一站式**,团队协作顺畅;③底层 Delta Lake 给你 ACID、时间旅行(21.11)。但诚实地说:①**贵**——Databricks 在云资源之上加了自己的溢价;②**有锁定**——虽然 Delta Lake 和 Spark 是开源的(这点比纯专有系统好),但深度用 Unity Catalog、Databricks 专有优化、工作流后,迁移仍有成本。所以它适合**愿意为免运维和一体化付费、且规模够大**的团队。
3. **诚实的边界:平台是工具,数据工程的功力在别处**。①**Databricks/Medallion 不会替你想清楚"数据该怎么建模"**——分几层、每层放什么、silver 里到底要做哪些清洗校验、gold 要哪些聚合,这些**业务和数据建模的判断**才是真功夫,平台只是让你把想清楚的东西高效实现。②**别为小数据上 Databricks**:和 Part 21 反复强调的一样,几个 GB 的数据用 Polars/DuckDB 单机就秒杀,上 Databricks(Spark)是杀鸡用牛刀又贵;它的价值在真正的大规模、多团队协作、需要托管湖仓的场景。③**Databricks vs Snowflake**(下节)是当前数据平台的两大阵营:Databricks 从"数据湖 + Spark + ML"出发(更代码化、更适合数据工程和 ML),Snowflake 从"SQL 数仓"出发(更易用、更适合分析师)——两者在互相靠拢(Databricks 加强 SQL,Snowflake 加强 Python/ML),选型看团队技能栈和主要负载。**结论:Databricks 是把 Spark/Delta/MLflow/治理打包成一体化托管湖仓平台, 核心最佳实践是 Medallion 架构(bronze 原始→silver 清洗→gold 聚合)带来的分层纪律——这套思想通用于所有数据工程; 但平台只是高效实现工具, 真功夫在数据建模判断; 且它贵、有锁定、只对大规模值得, 小数据仍用单机工具。**

**English**:
1. **The Medallion architecture's value is "turning a chaotic data pipeline into disciplined engineering"**: real-world data pipelines easily become spaghetti — raw data cleaned, aggregated, re-cleaned arbitrarily across scripts, logic scattered, problems untraceable, and someone else's cleaning too untrustworthy to reuse. Medallion solves it with three simple disciplines: ① **bronze faithfully keeps raw** (always traceable, replayable), ② **silver centralizes cleaning/validation once** (all downstreams share one clean copy, no re-cleaning), ③ **gold aggregates per business** (directly usable). Our experiment clearly showed this "layer-by-layer refinement": 7 dirty rows with null/duplicate/negative/type-mess became 4 clean trustworthy records in silver, then business aggregates in gold. **This layering is not unique to Databricks but a universal modern-data-engineering best practice** — even without Databricks, organize this way with Spark/dbt/Airflow.
2. **Databricks's value is "integration + no ops," at the cost of "vendor premium + some lock-in"**: it packages the open-source tools you learned in Parts 21/22 (Spark, Delta Lake, MLflow) into a managed platform on your cloud, with real benefits: ① **no self-built Spark cluster, tuning, or ops** (saving a whole platform team's work); ② **data (Delta) + compute (Spark) + ML (MLflow) + governance (Unity Catalog) + collaboration (notebooks) in one place**, smooth team collaboration; ③ underlying Delta Lake gives you ACID and time travel (21.11). But honestly: ① **expensive** — Databricks adds its own premium on top of cloud resources; ② **some lock-in** — although Delta Lake and Spark are open source (better than pure-proprietary systems), deep use of Unity Catalog, Databricks proprietary optimizations, and workflows still makes migration costly. So it suits teams **willing to pay for no-ops and integration, and large enough in scale**.
3. **Honest limits: the platform is a tool; the skill of data engineering lies elsewhere**. ① **Databricks/Medallion won't figure out "how the data should be modeled" for you** — how many layers, what goes in each, exactly what cleaning/validation silver does, which aggregates gold needs — these **business and data-modeling judgments** are the real skill; the platform just efficiently implements what you've thought through. ② **Don't use Databricks for small data**: as Part 21 stressed, a few GB is crushed by single-machine Polars/DuckDB, and Databricks (Spark) is an expensive sledgehammer; its value is in true large scale, multi-team collaboration, needing a managed lakehouse. ③ **Databricks vs Snowflake** (next section) are the two camps of current data platforms: Databricks starts from "data lake + Spark + ML" (more code-oriented, better for data engineering and ML), Snowflake from "SQL warehouse" (easier, better for analysts) — the two are converging (Databricks strengthening SQL, Snowflake strengthening Python/ML), so choose by team skill set and primary workload. **Conclusion: Databricks packages Spark/Delta/MLflow/governance into an integrated managed lakehouse platform, and its core best practice is the Medallion architecture (bronze raw → silver clean → gold aggregate), bringing layering discipline universal to all data engineering; but the platform is just an efficient implementation tool — the real skill is data-modeling judgment; and it's expensive, has some lock-in, and is worthwhile only at scale, with small data still using single-machine tools.**

> 💼 **实战视角 / Practical angle**
> **中文**:Databricks 落地:①**用 Medallion 组织数据**——bronze(原始 Delta 表, 追加式)→silver(清洗/去重/校验/join)→gold(业务聚合);这套思想即使不用 Databricks 也照用(Spark/dbt/Airflow);②**Delta Lake 做存储层**(ACID/时间旅行/MERGE, 接 21.11), 定期 OPTIMIZE 合并小文件、VACUUM 清历史;③**Unity Catalog** 做权限/血缘/数据发现(接 23.3 的 RBAC 思想);④**Workflows** 编排管道(接 23.6);⑤**MLflow** 追踪实验、注册模型(22.8);⑥成本上开 Photon 加速、自动伸缩集群、别让集群空转。**选型**:大规模湖仓+ML+数据工程→Databricks; 纯 SQL 分析+易用→Snowflake(下节); 小数据→单机 Polars/DuckDB。面试金句:*"Databricks 是 Spark 团队的统一湖仓平台(托管 Spark+Delta+MLflow+治理); 核心是 Medallion 架构 bronze(原始)→silver(清洗校验)→gold(业务聚合)逐层提质, 好处是职责清晰/可追溯重算/复用, 这套分层思想通用于所有数据工程; 底层 Delta Lake 给 ACID 时间旅行、Unity Catalog 做治理; 但它贵有锁定, 只对大规模值得。"*
> **English**: Databricks in practice: ① **organize data with Medallion** — bronze (raw Delta tables, append-only) → silver (clean/dedupe/validate/join) → gold (business aggregates); use this idea even without Databricks (Spark/dbt/Airflow); ② **Delta Lake as the storage layer** (ACID/time travel/MERGE, per 21.11), periodically OPTIMIZE to compact small files and VACUUM to clean history; ③ **Unity Catalog** for permissions/lineage/data discovery (per 23.3's RBAC idea); ④ **Workflows** to orchestrate pipelines (per 23.6); ⑤ **MLflow** to track experiments and register models (22.8); ⑥ for cost, enable Photon acceleration, autoscale clusters, don't let clusters idle. **Tool choice**: large-scale lakehouse + ML + data engineering → Databricks; pure SQL analytics + ease → Snowflake (next section); small data → single-machine Polars/DuckDB. Interview line: *"Databricks is the Spark team's unified lakehouse platform (managed Spark + Delta + MLflow + governance); its core is the Medallion architecture bronze (raw) → silver (clean/validate) → gold (business aggregate), progressively improving quality with benefits of clear responsibility/traceability/reuse, a layering idea universal to all data engineering; underneath, Delta Lake gives ACID and time travel, Unity Catalog does governance; but it's expensive with some lock-in, worthwhile only at scale."*

---
### 小结 / Summary
- **中文**:Databricks=Spark 团队的一体化托管湖仓平台(Spark+Delta Lake+MLflow+治理), 跑在你的云上。
- **English**: Databricks = the Spark team's integrated managed lakehouse platform (Spark + Delta Lake + MLflow + governance), on your cloud.
- **中文**:核心最佳实践=Medallion 架构:bronze(原始摄取)→silver(清洗/去重/校验)→gold(业务聚合), 逐层提升数据质量。
- **English**: Core best practice = Medallion architecture: bronze (raw ingest) → silver (clean/dedupe/validate) → gold (business aggregate), progressively improving quality.
- **中文**:分层通用于所有数据工程(职责清晰/可追溯/复用); 平台省运维但贵有锁定, 小数据仍用单机工具。
- **English**: Layering is universal to all data engineering (clear responsibility/traceability/reuse); the platform saves ops but is expensive with lock-in — small data still uses single-machine tools.
